# 07 · Window functions

**Dataset:** 1,000 synthetic healthcare claims (January–June 2026). All data is fake.

| Table | Columns |
|---|---|
| `claims` (1,000) | `claim_id`, `patient_id`, `provider_id`, `date_of_service`, `cpt_code`, `billed_amount`, `paid_amount`, `claim_status`, `denial_reason` |
| `patients` (200) | `patient_id`, `patient_name`, `date_of_birth`, `insurance_plan`, `state` |
| `providers` (13) | `provider_id`, `provider_name`, `specialty`, `state` |
| `procedures` (13) | `cpt_code`, `description`, `service_type`, `standard_charge` |

> 🇪🇸 **Para Josué:** este tema ya lo sabes (lo viste el 13/11 con los 10 claims). Es el tema que **más preguntan en entrevistas**.
> **Te recuerdo la idea clave:** `GROUP BY` **junta** filas; una función de ventana **calcula sin juntar** (cada claim sigue en su fila y gana una columna nueva).
> Dentro de `OVER (...)`: `PARTITION BY` = **qué grupos** (vuelve a empezar en cada grupo) y `ORDER BY` = **en qué orden**.
> La primera vez, corre todo con **Run All**. **Guarda con Ctrl + S después de cada ejercicio.**

In [2]:
# Load the CSV files as tables.
import duckdb

duckdb.sql("CREATE OR REPLACE TABLE claims     AS SELECT * FROM '../data/claims.csv'")
duckdb.sql("CREATE OR REPLACE TABLE patients   AS SELECT * FROM '../data/patients.csv'")
duckdb.sql("CREATE OR REPLACE TABLE providers  AS SELECT * FROM '../data/providers.csv'")
duckdb.sql("CREATE OR REPLACE TABLE procedures AS SELECT * FROM '../data/procedures.csv'")
print("Tables loaded: claims, patients, providers, procedures")

Tables loaded: claims, patients, providers, procedures


---
## Example · ROW_NUMBER

Number the claims of provider **PR001** from the most to the least expensive.

> 🇪🇸 **Te recuerdo:** `ROW_NUMBER()` pone 1, 2, 3... en el orden que dice el `ORDER BY` de adentro del `OVER`. Se escribe **con guion bajo** y con los paréntesis **pegados**.

In [3]:
duckdb.sql("""
    SELECT claim_id, billed_amount,
           ROW_NUMBER() OVER (ORDER BY billed_amount DESC) AS row_num
    FROM claims
    WHERE provider_id = 'PR001'
    ORDER BY row_num
    LIMIT 5
""").df()

,claim_id,billed_amount,row_num
0,CL00484,323,1
1,CL00686,318,2
2,CL00271,317,3
3,CL00567,315,4
4,CL00734,310,5


## Exercise 1 · Visit number of a patient

Number the visits of patient **PA0002** in the order they happened (oldest first). Show the result oldest first.

Columns: `claim_id`, `date_of_service`, `cpt_code`, `visit_number`.

✔ **12 rows.** The first one is **CL00157** · 2026-01-27 · 74177 · **1**. The last one is **CL00928** · 2026-06-17 · 99215 · **12**.

> 🇪🇸 Como el ejemplo, pero el `ORDER BY` de adentro del `OVER` es por **fecha**, y el `WHERE` filtra al paciente. Ojo: hay **dos** `ORDER BY`, el de adentro del `OVER` (cómo numerar) y el del final (cómo mostrar).

In [4]:
duckdb.sql("""
    SELECT claim_id, date_of_service, cpt_code,
    ROW_NUMBER() OVER(ORDER BY date_of_service) AS visit_number
    FROM claims
    WHERE patient_id = 'PA0002' 
""").df()

,claim_id,date_of_service,cpt_code,visit_number
0,CL00157,2026-01-27,74177,1
1,CL00172,2026-01-28,72148,2
2,CL00183,2026-01-30,93000,3
3,CL00301,2026-02-21,97140,4
4,CL00374,2026-03-07,71046,5
5,CL00495,2026-03-26,85025,6
6,CL00587,2026-04-11,99215,7
7,CL00804,2026-05-25,93000,8
8,CL00873,2026-06-08,97140,9
9,CL00875,2026-06-09,72148,10


---
## Exercise 2 · Rank the providers by total billed

Using a CTE named `provider_totals` with the total billed by each `provider_id`, rank the providers from the highest to the lowest total billed. Show them in rank order.

Columns: `provider_id`, `total_billed`, `billing_rank`.

✔ **14 rows** (12 providers with claims, plus the 2 orphan codes PR099 and PR104). The first one is **PR002** · $96,838 · **1**. The last one is **PR099** · $520 · **14**.

> 🇪🇸 **Te recuerdo** el CTE del tema 6:
> ```sql
> WITH provider_totals AS (
>     SELECT provider_id, SUM(billed_amount) AS total_billed
>     FROM claims
>     GROUP BY provider_id
> )
> ```
> Afuera: `RANK() OVER (ORDER BY total_billed DESC) AS billing_rank`.
> `RANK` es como `ROW_NUMBER`, pero si dos empatan les da **el mismo puesto** (como en una carrera).
> Por qué el CTE: primero **juntas** (un total por proveedor) y después **rankeas** esos totales. Dos pasos, como dos hojas de Excel.

In [22]:
duckdb.sql("""
    WITH provider_totals AS(
    SELECT provider_id, SUM(billed_amount) AS total_billed
    FROM claims
    GROUP BY provider_id)
    SELECT
    provider_id, total_billed,
    RANK() OVER(ORDER BY total_billed DESC) AS billing_rank
    FROM provider_totals
    ORDER BY  RANK() OVER(ORDER BY total_billed DESC)
    
""").df()

,provider_id,total_billed,billing_rank
0,PR002,96838.0,1
1,PR011,93578.0,2
2,PR007,75206.0,3
3,PR008,32008.0,4
4,PR003,29421.0,5
5,PR009,18844.0,6
6,PR004,15466.0,7
7,PR012,12979.0,8
8,PR006,12921.0,9
9,PR005,11885.0,10


---
## Example · SUM with OVER

Each claim of **PR001** next to the **total** of its provider.

> 🇪🇸 **Te recuerdo:** `SUM(...) OVER (PARTITION BY ...)` **sin** `ORDER BY` adentro da el **total del grupo**, repetido en cada fila. No junta filas: cada claim sigue en su fila.

In [6]:
duckdb.sql("""
    SELECT claim_id, billed_amount,
           SUM(billed_amount) OVER (PARTITION BY provider_id) AS provider_total
    FROM claims
    WHERE provider_id = 'PR001'
    LIMIT 5
""").df()

,claim_id,billed_amount,provider_total
0,CL00045,91,11582.0
1,CL00065,222,11582.0
2,CL00081,95,11582.0
3,CL00111,158,11582.0
4,CL00114,288,11582.0


## Exercise 3 · Each claim as a percentage of its provider total

For the claims of provider **PR001**, show each claim, the provider total, and what **percentage of the provider total** that claim is (one decimal). Sort from the most to the least expensive claim.

Columns: `claim_id`, `billed_amount`, `provider_total`, `pct_of_provider`.

✔ **76 rows.** The first one is **CL00484** · 323 · 11,582 · **2.8**.

> 🇪🇸 `pct_of_provider` = el claim ÷ el total del proveedor × 100. Es la parte ÷ el todo, como en el reto del tema 6:
> ```sql
> ROUND(100.0 * billed_amount / SUM(billed_amount) OVER (PARTITION BY provider_id), 1)
> ```
> Aquí el "todo" no viene de una subconsulta sino del `OVER`. Mismo resultado, otra herramienta.

In [7]:
duckdb.sql("""
    SELECT claim_id, billed_amount,
    SUM(billed_amount) OVER (PARTITION BY provider_id) AS provider_total,
    ROUND(100.0*billed_amount/SUM(billed_amount) OVER (PARTITION BY provider_id),1) AS pct_of_provider
    FROM claims
    WHERE provider_id = 'PR001'
    ORDER BY billed_amount DESC
""").df()

,claim_id,billed_amount,provider_total,pct_of_provider
0,CL00484,323,11582.0,2.8
1,CL00686,318,11582.0,2.7
2,CL00271,317,11582.0,2.7
3,CL00567,315,11582.0,2.7
4,CL00734,310,11582.0,2.7
...,...,...,...,...
71,CL00273,59,11582.0,0.5
72,CL00500,59,11582.0,0.5
73,CL00901,58,11582.0,0.5
74,CL00596,57,11582.0,0.5


---
## Example · LAG

The date of each claim of patient **PA0002** next to the date of the **previous** claim.

> 🇪🇸 **Te recuerdo:** `LAG(columna)` trae el valor de **la fila anterior**, en el orden del `OVER`. La primera fila no tiene anterior: sale vacía (`NaT` o `NA`).

In [8]:
duckdb.sql("""
    SELECT claim_id, date_of_service,
           LAG(date_of_service) OVER (ORDER BY date_of_service) AS previous_date
    FROM claims
    WHERE patient_id = 'PA0002'
    ORDER BY date_of_service
""").df()

,claim_id,date_of_service,previous_date
0,CL00157,2026-01-27,NaT
1,CL00172,2026-01-28,2026-01-27
2,CL00183,2026-01-30,2026-01-28
3,CL00301,2026-02-21,2026-01-30
4,CL00374,2026-03-07,2026-02-21
5,CL00495,2026-03-26,2026-03-07
6,CL00587,2026-04-11,2026-03-26
7,CL00804,2026-05-25,2026-04-11
8,CL00873,2026-06-08,2026-05-25
9,CL00875,2026-06-09,2026-06-08


## Exercise 4 · Days between visits

For patient **PA0002**, show how many **days** passed since the previous visit. Show the result oldest first.

Columns: `claim_id`, `date_of_service`, `days_since_last`.

✔ **12 rows.** The first one is empty (no previous visit). **CL00301** = **22** days. The longest gap is **CL00804** = **44** days.

> 🇪🇸 Restar fechas da días (lo viste el martes): `date_of_service - LAG(date_of_service) OVER (...)`. La fecha **más nueva** va primero.

In [28]:
duckdb.sql("""
    SELECT claim_id, date_of_service,
    date_of_service - LAG(date_of_service) OVER(ORDER BY date_of_service) AS days_since_last
    FROM claims
    WHERE patient_id = 'PA0002'
    ORDER BY date_of_service
""").df()

,claim_id,date_of_service,days_since_last
0,CL00157,2026-01-27,<NA>
1,CL00172,2026-01-28,1
2,CL00183,2026-01-30,2
3,CL00301,2026-02-21,22
4,CL00374,2026-03-07,14
5,CL00495,2026-03-26,19
6,CL00587,2026-04-11,16
7,CL00804,2026-05-25,44
8,CL00873,2026-06-08,14
9,CL00875,2026-06-09,1


---
## Exercise 5 · The most expensive claim of each provider ⭐⭐

Show the **most expensive claim of each provider**, with the provider name. Sort from the most to the least expensive, and if two tie, by `provider_name` (A to Z).

Columns: `provider_name`, `claim_id`, `billed_amount`.

✔ **12 rows.** The first one is **Lakeside Imaging** · CL00756 · **1,650**. The second one is **Desert Sun Radiology** · CL00880 · 1,648.

> 🇪🇸 **Te recuerdo** el truco del "top 1 por grupo" (13/11): CTE con `ROW_NUMBER` + afuera `WHERE fila = 1`.
> 1. CTE `ranked`: `provider_id`, `claim_id`, `billed_amount` y
>    `ROW_NUMBER() OVER (PARTITION BY provider_id ORDER BY billed_amount DESC, claim_id) AS row_num`.
> 2. Afuera: `JOIN` con `providers` (para el nombre) y `WHERE row_num = 1`.
>
> **¿Por qué `, claim_id` dentro del `OVER`?** Dos proveedores tienen **empates** (dos claims con el mismo monto máximo). Sin desempate, `ROW_NUMBER` elige uno al azar y el resultado puede cambiar cada vez. Con `claim_id` se queda siempre con el código más bajo.
> **¿Por qué 12 y no 14?** Con `JOIN` se van los huérfanos PR099 y PR104 (no tienen nombre). PR013 no tiene claims.

In [26]:
duckdb.sql("""
    WITH ranked AS (
    SELECT provider_id, claim_id, billed_amount,
    ROW_NUMBER() OVER (PARTITION BY provider_id ORDER BY billed_amount DESC, claim_id ASC) AS row_num
    FROM claims)
    SELECT pr.provider_name, r.claim_id, r.billed_amount
    FROM ranked as r
    INNER JOIN providers as pr
    ON r.provider_id = pr.provider_id
    WHERE row_num = 1
    ORDER BY billed_amount DESC, provider_name ASC
""").df()

,provider_name,claim_id,billed_amount
0,Lakeside Imaging,CL00756,1650
1,Desert Sun Radiology,CL00880,1648
2,Empire Diagnostic Imaging,CL00451,1648
3,Golden Coast Heart Center,CL00177,1044
4,Northside Cardiology,CL00248,1035
5,Pine Valley Medical Group,CL00219,330
6,Lone Star Internal Medicine,CL00700,327
7,Riverside Family Practice,CL00357,327
8,Maple Street Clinic,CL00262,324
9,Sunrise Family Clinic,CL00484,323


---
## Exercise 6 · Running count of denials ⭐

Provider **PR007** (Desert Sun Radiology) has the highest denial rate. Show its claims in date order with a **running count of denied claims**. Show the result in date order too.

Columns: `claim_id`, `date_of_service`, `claim_status`, `running_denials`.

✔ **72 rows.** The first 3: CL00032 · Denied · **1** / CL00047 · Paid · **1** / CL00052 · Denied · **2**. The last row says **24**.

> 🇪🇸 **Te recuerdo** el acumulado: `SUM(...) OVER (ORDER BY ...)` **con** `ORDER BY` adentro va sumando fila por fila.
> Para contar solo los Denied, suma un `CASE WHEN` que pone 1 o 0 (como en la tasa de rechazo):
> ```sql
> SUM(CASE WHEN claim_status = 'Denied' THEN 1 ELSE 0 END) OVER (ORDER BY date_of_service, claim_id) AS running_denials
> ```
> **¿Por qué `, claim_id`?** Hay claims del mismo día (CL00047 y CL00052 son del 9 de enero). Sin desempate, SQL suma los dos juntos y la fila de CL00047 ya diría 2.
> En el `ORDER BY` del final usa las mismas dos columnas.

In [16]:
duckdb.sql("""
    SELECT claim_id, date_of_service, claim_status,
    SUM( CASE WHEN claim_status = 'Denied' THEN 1 ELSE 0 END) OVER (ORDER BY date_of_service, claim_id) AS running_denials
    FROM claims
    WHERE provider_id = 'PR007' 
    ORDER BY date_of_service
    
""").df()

,claim_id,date_of_service,claim_status,running_denials
0,CL00032,2026-01-06,Denied,1.0
1,CL00047,2026-01-09,Paid,1.0
2,CL00052,2026-01-09,Denied,2.0
3,CL00056,2026-01-10,Paid,2.0
4,CL00058,2026-01-11,Denied,3.0
...,...,...,...,...
67,CL00880,2026-06-09,Paid,22.0
68,CL00899,2026-06-12,Paid,22.0
69,CL00929,2026-06-18,Denied,23.0
70,CL00943,2026-06-20,Paid,23.0


---
## Challenge ⭐⭐⭐ · Monthly billing trend

For each month, show the total billed, the **running total** since January, and the **change** versus the previous month. Sort by month.

Columns: `service_month`, `total_billed`, `running_billed`, `change_vs_prev`.

✔ **6 rows:**

| service_month | total_billed | running_billed | change_vs_prev |
|---|---|---|---|
| 2026-01-01 | 85,499 | 85,499 | *(empty)* |
| 2026-02-01 | 73,256 | 158,755 | −12,243 |
| 2026-03-01 | 71,798 | 230,553 | −1,458 |
| 2026-04-01 | 69,244 | 299,797 | −2,554 |
| 2026-05-01 | 65,076 | 364,873 | −4,168 |
| 2026-06-01 | 60,212 | 425,085 | −4,864 |

> 🇪🇸 Junta tres cosas que ya sabes:
> 1. **CTE `monthly`:** el total por mes. **Te recuerdo** `DATE_TRUNC` (martes): `DATE_TRUNC('month', date_of_service) AS service_month`, `SUM(billed_amount) AS total_billed` y `GROUP BY DATE_TRUNC('month', date_of_service)`.
> 2. **Afuera, el acumulado:** `SUM(total_billed) OVER (ORDER BY service_month)`.
> 3. **Afuera, el cambio:** `total_billed - LAG(total_billed) OVER (ORDER BY service_month)`.
>
> **El hallazgo:** lo facturado **baja todos los meses**. En un reporte real eso se investiga: ¿menos pacientes, menos proveedores, o claims que todavía no se cargan?

In [27]:
duckdb.sql("""
    WITH monthly AS (
    SELECT DATE_TRUNC('month', date_of_service) AS service_month,
    SUM (billed_amount) AS total_billed
    FROM claims
    GROUP BY DATE_TRUNC('month', date_of_service)
    )
    SELECT service_month, total_billed, SUM (total_billed) OVER (ORDER BY service_month) AS running_billed,  
    total_billed - LAG(total_billed) OVER (ORDER BY service_month) AS change_vs_prev
    FROM monthly
    ORDER BY service_month
    
""").df()

,service_month,total_billed,running_billed,change_vs_prev
0,2026-01-01,85499.0,85499.0,NaN
1,2026-02-01,73256.0,158755.0,-12243.0
2,2026-03-01,71798.0,230553.0,-1458.0
3,2026-04-01,69244.0,299797.0,-2554.0
4,2026-05-01,65076.0,364873.0,-4168.0
5,2026-06-01,60212.0,425085.0,-4864.0


---
## Check your understanding

> 🇪🇸 Responde en el chat, con tus palabras:
> 1. ¿Qué diferencia hay entre `ROW_NUMBER` y `RANK` cuando dos claims tienen el mismo monto?
> 2. En el ejercicio 3, `SUM(billed_amount) OVER (PARTITION BY provider_id)` da el mismo número que un `GROUP BY provider_id`. Entonces, ¿por qué aquí no se puede usar `GROUP BY`?